# LC 134 — Gas Station
**Difficulty:** Medium &nbsp;|&nbsp; **Category:** Greedy
**Pattern:** Total Tank Check + Local Tank Reset

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> If total gas >=
total cost, a solution always exists. The starting
station is the one after the point where the
running tank first goes negative — greedy resets
the start there.
</div>

## Official Problem Statement

There are `n` gas stations along a circular route.
You are given two integer arrays `gas` and `cost`
where `gas[i]` is the gas at station `i` and
`cost[i]` is the gas to travel from station `i`
to station `i+1`.

Return the starting gas station's index if you
can travel around the circuit once in the
clockwise direction, otherwise return `-1`.
If a solution exists, it is **guaranteed unique**.

**Example 1:**
```
Input:  gas=[1,2,3,4,5], cost=[3,4,5,1,2]
Output: 3
```
**Example 2:**
```
Input:  gas=[2,3,4], cost=[3,4,3]
Output: -1
```

**Constraints:**
- `n == gas.length == cost.length`
- `1 <= n <= 10^5`
- `0 <= gas[i], cost[i] <= 10^4`

## What This Is Actually Asking

Gas stations sit in a circle. Each station gives
you some fuel and costs fuel to reach the next.
Find a starting station from which you can make
a full loop without running dry. Return -1 if
no such station exists.

## Walk Through an Example by Hand

```
gas  = [1, 2, 3, 4, 5]
cost = [3, 4, 5, 1, 2]
diff = [-2,-2,-2, 3, 3]   (gain at each station)

total = sum(diff) = 0 >= 0 -> solution exists

tank=0  start=0

i=0: tank = 0 + (-2) = -2  < 0 -> reset
     start=1  tank=0
i=1: tank = 0 + (-2) = -2  < 0 -> reset
     start=2  tank=0
i=2: tank = 0 + (-2) = -2  < 0 -> reset
     start=3  tank=0
i=3: tank = 0 + 3 = 3   >= 0  ok
i=4: tank = 3 + 3 = 6   >= 0  ok

Answer: start = 3
Verify: start at 3, tank=4-1=3, +5-2=6, +1-3=4,
        +2-4=2, +3-5=0. Completes the loop.
```

## The Picture

```
gas  = [1, 2, 3, 4, 5]
cost = [3, 4, 5, 1, 2]
diff = [-2,-2,-2, 3, 3]

Running tank from start=0:
  0   -2   -4   -6   -3   0
      ^-- goes negative immediately

Key insight 1:
  If total gas < total cost -> impossible -> -1
  (not enough fuel in total to complete the circuit)

Key insight 2:
  If running tank goes negative starting at S,
  then S cannot be the start, AND
  neither can any station between S and the
  point where tank went negative.
  Why? Starting from any intermediate point would
  also run dry at the same spot (arrived with less).
  -> Reset start to i+1, reset tank to 0.

One pass finds the answer.
```

## When To Use This Pattern

- When asked for **circular route feasibility**,
  think **total check first: sum(gas) >= sum(cost)**
- When the running tank goes negative at i, think
  **none of start..i can work — reset start to i+1**
- When total is non-negative and loop completes,
  think **the last reset start is the answer**
- When total is negative, think **return -1 early**

## The Approach

Compute the total surplus (total gas minus total
cost). If negative, return -1 immediately.
Otherwise walk through the stations with a running
tank. Whenever the tank goes negative, the current
start is invalid — reset the start to the next
station and the tank to zero. Return the final
start after the loop completes.

In [ ]:
from typing import List  # type hints for the solution

In [ ]:
def test_harness(func):
    tests = [
        # (gas, cost, expected)
        ([1,2,3,4,5], [3,4,5,1,2],  3),
        ([2,3,4],     [3,4,3],      -1),
        ([5],         [4],           0),   # single station
        ([1,2],       [2,1],         1),
        ([3,3,4],     [3,4,3],      -1),
        ([4,5,2,6,5,3],[3,2,7,3,2,9], 3),
        ([2,0,1,2,3,4],[0,1,0,0,0,0], 0),
    ]

    passed = 0
    for i, (gas, cost, expected) in enumerate(tests):
        result = func(gas[:], cost[:])
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(
            f"Test {i+1}: {status} | "
            f"gas={gas} cost={cost} | "
            f"expected={expected} | got={result}"
        )

    print(f"\n{passed}/{len(tests)} tests passed")

In [ ]:
def canCompleteCircuit(
    gas: List[int], cost: List[int]
) -> int:
    """
    Return start index for complete circuit, or -1.

    If sum(gas) < sum(cost): return -1. Otherwise
    walk with running tank; when tank < 0 reset
    start = i+1, tank = 0. Return start at end.

    Time:  O(n) — single pass
    Space: O(1) — three variables
    """
    pass


# Quick debug — run this cell while building
print(canCompleteCircuit([1,2,3,4,5],[3,4,5,1,2]))  # 3
print(canCompleteCircuit([2,3,4],[3,4,3]))            # -1
print(canCompleteCircuit([5],[4]))                    # 0
print(canCompleteCircuit([1,2],[2,1]))                # 1

In [ ]:
# Uncomment and run when solution is ready
# test_harness(canCompleteCircuit)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Brute force — try every start | O(n²) | O(1) |
| Greedy single pass | O(n) | O(1) |

The greedy proof: when tank drops below zero from
start S through station i, no station in [S..i]
can be the answer. This eliminates O(n) candidates
in O(1) work at each drop — giving linear time.

## Real World Connection

At Citi, the nightly batch pipeline is a circular
workflow: each stage consumes and produces data
tokens. The 'gas station' question determines
which stage can safely begin the daily run without
stalling midway due to insufficient data tokens.
The single-pass greedy identifies the valid start
stage in O(n) — far faster than the O(n²) brute
force that was previously re-simulating from
every possible start point.
On AWS, the same greedy models DLQ (dead-letter
queue) replay: find the first message in the
batch from which replay can complete without
hitting a throttle limit.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra